In [0]:
df_emp = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/employees")
df_dept = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/departments")
df_hires = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/new_hires")
df_bands = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/salary_bands")   

display(df_emp)
display(df_dept)
display(df_hires)
display(df_bands)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5
4,David,HR,61000,2022-01-10,M,27,true,6100,4
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3


dept,location,manager,budget
Engineering,New York,Sara,500000
Marketing,Chicago,Tom,300000
HR,Austin,Asha,200000
Finance,Dallas,Leo,400000


id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3


dept,gender,band_label
Engineering,F,Band-A
Engineering,M,Band-A
Marketing,F,Band-B
Marketing,M,Band-B
HR,F,Band-C
HR,M,Band-C
Finance,F,Band-B
Finance,M,Band-B


In [0]:
display(
    df_emp.join(df_dept, on='dept', how='inner')
    .select(
        df_emp["*"],
        df_dept["location"],
        df_dept["manager"],
        df_dept["budget"]
    )
)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,location,manager,budget
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4,New York,Sara,500000
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3,Chicago,Tom,300000
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5,New York,Sara,500000
4,David,HR,61000,2022-01-10,M,27,true,6100,4,Austin,Asha,200000
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4,Chicago,Tom,300000
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5,New York,Sara,500000
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3,Austin,Asha,200000


In [0]:
display(
    df_emp.join(df_dept, on='dept', how='left')
    .filter(df_dept["location"].isNull())
)

dept,id,name,salary,join_date,gender,age,is_active,bonus,rating,location,manager,budget


In [0]:
display(
    df_dept.join(df_emp, on='dept', how='left_anti')
)

dept,location,manager,budget
Finance,Dallas,Leo,400000


In [0]:
display(
    df_emp.join(df_dept, on='dept', how='inner')
    .join(df_bands, (df_emp["dept"] == df_bands["dept"]) & (df_emp["gender"] == df_bands["gender"]), how='inner')
    .select(
        df_emp["name"],
        df_emp["dept"],
        df_dept["location"],
        df_dept["manager"],
        df_bands["band_label"]
    )
)

name,dept,location,manager,band_label
Alice,Engineering,New York,Sara,Band-A
Bob,Marketing,Chicago,Tom,Band-B
Carol,Engineering,New York,Sara,Band-A
David,HR,Austin,Asha,Band-C
Eve,Marketing,Chicago,Tom,Band-B
Frank,Engineering,New York,Sara,Band-A
Grace,HR,Austin,Asha,Band-C


In [0]:
display(
    df_emp.join(df_dept, on='dept', how='inner')
    .join(df_bands, (df_emp["dept"] == df_bands["dept"]) & (df_emp["gender"] == df_bands["gender"]), how='inner').select(
        df_emp["name"],
        df_emp["dept"],
        df_dept["location"],
        df_dept["manager"],
        df_bands["band_label"]
    )
)

name,dept,location,manager,band_label
Alice,Engineering,New York,Sara,Band-A
Bob,Marketing,Chicago,Tom,Band-B
Carol,Engineering,New York,Sara,Band-A
David,HR,Austin,Asha,Band-C
Eve,Marketing,Chicago,Tom,Band-B
Frank,Engineering,New York,Sara,Band-A
Grace,HR,Austin,Asha,Band-C


In [0]:
from pyspark.sql.functions import col

emp_a = df_emp.alias("a")
emp_b = df_emp.alias("b")

display(
    emp_a.join(emp_b, on='dept', how='inner')
    .filter(col("a.name") < col("b.name")) 
    .select(
        col("a.name").alias("emp1"),
        col("b.name").alias("emp2"),
        col("a.dept").alias("dept")
    )
)

emp1,emp2,dept
Alice,Frank,Engineering
Bob,Eve,Marketing
Carol,Frank,Engineering
David,Grace,HR
Alice,Carol,Engineering
